# MintyPython Demo

This notebook demonstrates all major features of MintyPython (Model INTerpretation with pYthon) - a visualization library for comparing GLM and GBM models.

**Contents:**
1. Setup & Data Generation
2. Initialize MintyPython
3. SHAP Value Computation
4. Univariate Plots
5. Bivariate Plots
6. Model Comparison
7. Configuration Customization
8. Plot Engines (Bokeh vs Matplotlib)

## 1. Setup & Data Generation

First, let's import the required packages and generate synthetic insurance data.

In [1]:
# Standard imports
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Bokeh setup for notebook display
from bokeh.io import output_notebook
output_notebook()

# Import demo modules
from synthetic_data import generate_synthetic_data, train_xgboost_model, prepare_data_for_mintypython, encode_categoricals
from glm_helpers import get_glm_data_for_mintypython, create_category_mapping_dict

# Import MintyPython
from mintypython import mintypython

print("All imports successful!")

Loading BokehJS ...

All imports successful!


In [2]:
# Generate synthetic insurance data
raw_data = generate_synthetic_data(n_samples=10000, random_state=42)

print(f"Generated {len(raw_data):,} samples")
print(f"\nColumns: {list(raw_data.columns)}")
raw_data.head()

Generated 10,000 samples

Columns: ['age', 'vehicle_value', 'years_licensed', 'region', 'vehicle_type', 'exposure', 'claim_count']


,age,vehicle_value,years_licensed,region,vehicle_type,exposure,claim_count
0,52.450712,15689.588755,11.677637,North,SUV,0.639255,0
1,42.926035,18906.296023,24.677705,West,Sedan,0.946568,0
2,54.715328,16338.988603,31.575657,South,SUV,0.989004,0
3,67.845448,23276.720582,45.267958,South,Truck,0.839560,0
4,41.487699,40078.257759,18.461473,West,SUV,0.985150,0


In [3]:
raw_data.dtypes


age               float64
vehicle_value     float64
years_licensed    float64
region                str
vehicle_type          str
exposure          float64
claim_count         int64
dtype: object

In [4]:
# Define feature names
feature_names = ['age', 'vehicle_value', 'years_licensed', 'region', 'vehicle_type']

# Prepare data for modeling:
# - Automatically encodes object/string columns to int8/int16
# - Preserves original values in {col}_original columns
# - Use optimize=True to also downcast numeric columns
data, category_mappings = prepare_data_for_mintypython(raw_data, feature_names, optimize=True)

print("Category mappings (code -> label):")
for col, mapping in category_mappings.items():
    print(f"  {col}: {mapping}")

print(f"\nOriginal categorical columns preserved:")
print(data[['region', 'region_original', 'vehicle_type', 'vehicle_type_original']].head())

Encoded 2 categorical column(s): ['region', 'vehicle_type']
Original values preserved in: ['region_original', 'vehicle_type_original']
Optimized 4 numeric column(s):
  age: float64 -> float32
  years_licensed: float64 -> float32
  exposure: float64 -> float32
  claim_count: int64 -> int8
Memory: 635.4KB -> 244.4KB (61.5% reduction)
Category mappings (code -> label):
  region: {0: 'East', 1: 'North', 2: 'South', 3: 'West'}
  vehicle_type: {0: 'SUV', 1: 'Sedan', 2: 'Sports', 3: 'Truck'}

Original categorical columns preserved:
   region region_original  vehicle_type vehicle_type_original
0       1           North             0                   SUV
1       3            West             1                 Sedan
2       2           South             0                   SUV
3       2           South             3                 Truck
4       3            West             0                   SUV


In [5]:
# Train XGBoost model with Poisson objective
model = train_xgboost_model(
    data,
    feature_names,
    weight_col='exposure',
    target_col='claim_count',
    random_state=42,
    max_depth=4,
    eta=0.1
)

print("Model trained successfully!")
print(f"Feature importance:")
importance = model.get_score(importance_type='gain')
for feat, score in sorted(importance.items(), key=lambda x: -x[1]):
    print(f"  {feat}: {score:.2f}")

Model trained successfully!
Feature importance:
  age: 2.06
  years_licensed: 1.42
  vehicle_value: 1.32
  vehicle_type: 1.15
  region: 1.05


## 2. Initialize MintyPython

Create a MintyPython instance with our model and data. We'll also set up simulated GLM relativities for comparison.

In [6]:
# Create simulated GLM relativities and predictions
glm_df, glm_preds = get_glm_data_for_mintypython(data, feature_names)

# Add GLM predictions to data
data['glm_predictions'] = glm_preds

print("GLM relativities created for:", list(glm_df.columns))
print(f"\nGLM predictions - Mean: {glm_preds.mean():.4f}, Actual Mean: {data['claim_count'].mean():.4f}")

GLM relativities created for: ['age', 'vehicle_value', 'years_licensed', 'region', 'vehicle_type']

GLM predictions - Mean: 0.0845, Actual Mean: 0.0489


In [7]:
# Create mapping dict for categorical labels in plots
mapping_dict = create_category_mapping_dict(category_mappings)

# Initialize MintyPython
mp = mintypython(
    data=data,
    model=model,
    weight_col='exposure',
    actuals_col='claim_count',
    feature_names=feature_names,
    link_fn='poisson',
    glm_preds_col='glm_predictions',
    glm_df=glm_df,
    mapping_dict=mapping_dict,
    verbose=True
)

print("MintyPython initialized!")

MintyPython initialized!


In [8]:
data.head()

,age,vehicle_value,years_licensed,region,vehicle_type,exposure,claim_count,region_original,vehicle_type_original,glm_predictions
0,52.450714,15689.588755,11.677636,1,0,0.639255,0.0,North,SUV,0.064795
1,42.926037,18906.296023,24.677706,3,1,0.946568,0.0,West,Sedan,0.084568
2,54.715328,16338.988603,31.575657,2,0,0.989004,0.0,South,SUV,0.075093
3,67.845451,23276.720582,45.267960,2,3,0.839560,0.0,South,Truck,0.036227
4,41.487698,40078.257759,18.461473,3,0,0.985150,0.0,West,SUV,0.118047


## 3. SHAP Value Computation

MintyPython computes SHAP values lazily when needed. Let's precompute them to see the process.

In [9]:
# Compute SHAP values (this is also done automatically when plotting)
mp.Data_prep.prep_shap_values()

print("SHAP values computed and cached.")
print(f"\nSHAP DataFrame shape: {mp.shap_df.shape}")
print(f"SHAP columns: {list(mp.shap_df.columns)}")

SHAP values computed and cached.

SHAP DataFrame shape: (10000, 5)
SHAP columns: ['age', 'vehicle_value', 'years_licensed', 'region', 'vehicle_type']


## 4. Univariate Plots

Univariate plots show how individual features affect model predictions.

### 4.1 Basic Univariate with SHAP

In [13]:
# Basic univariate plot showing SHAP values for age
mp.univariate_plot(
    var_name='age',
    shap=True,
    weight=True,
    plot_name='Age - Basic SHAP Plot'
)

figure(id='p1332', ...)

### 4.2 SHAP with Points and Standard Deviation

In [14]:
# SHAP with individual points and standard deviation bands
mp.univariate_plot(
    var_name='age',
    shap=True,
    shap_points=True,
    shap_sd=True,
    n_shap_points=500,
    weight=True,
    plot_name='Age - SHAP with Points and SD'
)

figure(id='p1427', ...)

### 4.3 SHAP with GLM Relativities Overlay

In [15]:
# Compare GBM SHAP values with GLM relativities
mp.univariate_plot(
    var_name='age',
    shap=True,
    glm=True,
    weight=True,
    plot_name='Age - SHAP vs GLM Comparison'
)

AttributeError: 'mintypython' object has no attribute 'emb_mdl'

### 4.4 Full Plot with Actuals

In [16]:
# Complete plot with SHAP, GLM, actuals, and weights
mp.univariate_plot(
    var_name='age',
    shap=True,
    glm=True,
    actuals=True,
    weight=True,
    plot_name='Age - Complete Analysis'
)

AttributeError: 'mintypython' object has no attribute 'emb_mdl'

### 4.5 Custom Binning

In [17]:
# Custom binning with start, finish, and stepsize
mp.univariate_plot(
    var_name='age',
    shap=True,
    weight=True,
    start=20,
    finish=70,
    stepsize=5,
    infinity_lower=True,
    infinity_higher=True,
    plot_name='Age - Custom Binning (20-70, step=5)'
)

figure(id='p1537', ...)

In [18]:
# Custom binning with nlevels (automatic step calculation)
mp.univariate_plot(
    var_name='vehicle_value',
    shap=True,
    glm=True,
    weight=True,
    nlevels=8,
    percentile_start=5,
    percentile_finish=95,
    plot_name='Vehicle Value - 8 Levels with Percentile Bounds'
)

AttributeError: 'mintypython' object has no attribute 'emb_mdl'

### 4.6 Categorical Variables

In [19]:
# Region - categorical variable
mp.univariate_plot(
    var_name='region',
    shap=True,
    glm=True,
    actuals=True,
    weight=True,
    plot_name='Region - Categorical Analysis'
)

AttributeError: 'mintypython' object has no attribute 'emb_mdl'

In [ ]:
# Vehicle type - categorical variable
mp.univariate_plot(
    var_name='vehicle_type',
    shap=True,
    glm=True,
    actuals=True,
    weight=True,
    plot_name='Vehicle Type - Categorical Analysis'
)

### 4.7 Rebasing to a Specific Level

In [ ]:
# Set a specific base level for relativities
mp.univariate_plot(
    var_name='vehicle_type',
    shap=True,
    glm=True,
    weight=True,
    base=1,  # Use Sedan (code 1) as base
    rebase=True,
    plot_name='Vehicle Type - Rebased to Sedan'
)

## 5. Bivariate Plots

Bivariate plots show interactions between two features.

### 5.1 Two Continuous Variables

In [20]:
# Age x Vehicle Value interaction
mp.bivariate_plot(
    var1='age',
    var2='vehicle_value',
    shap=True,
    nlevels_var1=6,
    nlevels_var2=4,
    plot_title='Age x Vehicle Value Interaction'
)

TypeError: category type does not support sum operations

### 5.2 Continuous x Categorical

In [21]:
# Age x Region interaction
mp.bivariate_plot(
    var1='age',
    var2='region',
    shap=True,
    nlevels_var1=8,
    plot_title='Age x Region Interaction'
)

KeyError: '0'

In [ ]:
# Vehicle Value x Vehicle Type interaction
mp.bivariate_plot(
    var1='vehicle_value',
    var2='vehicle_type',
    shap=True,
    nlevels_var1=6,
    plot_title='Vehicle Value x Vehicle Type Interaction'
)

### 5.3 Bivariate with GLM

In [ ]:
# Age x Region with GLM comparison
mp.bivariate_plot(
    var1='age',
    var2='region',
    shap=True,
    glm=True,
    nlevels_var1=6,
    plot_title='Age x Region - SHAP vs GLM'
)

## 6. Model Comparison

Compare multiple models using the `compare()` method.

In [ ]:
# Train a second model with different hyperparameters
model2 = train_xgboost_model(
    data,
    feature_names,
    weight_col='exposure',
    target_col='claim_count',
    random_state=123,
    max_depth=6,  # Deeper trees
    eta=0.05      # Lower learning rate
)

print("Second model trained!")

In [ ]:
# Create second MintyPython instance
mp2 = mintypython(
    data=data,
    model=model2,
    weight_col='exposure',
    actuals_col='claim_count',
    feature_names=feature_names,
    link_fn='poisson',
    mapping_dict=mapping_dict,
    verbose=False
)

print("Second MintyPython instance created!")

In [ ]:
# Compare models on age variable
mp.compare(
    mintylist=[mp2],
    mintynames=['Model 1 (depth=4)', 'Model 2 (depth=6)'],
    var_name='age',
    shap=True,
    weight=True,
    plot_name='Model Comparison - Age'
)

In [ ]:
# Compare models on vehicle_type
mp.compare(
    mintylist=[mp2],
    mintynames=['Model 1', 'Model 2'],
    var_name='vehicle_type',
    shap=True,
    weight=True,
    plot_name='Model Comparison - Vehicle Type'
)

## 7. Configuration Customization

Customize plot appearance through the `config` dictionary.

In [ ]:
# View default configuration
print("Default configuration:")
import json
print(json.dumps(mp.config, indent=2))

In [ ]:
# Create a copy of mp with custom colors
mp_custom = mintypython(
    data=data,
    model=model,
    weight_col='exposure',
    actuals_col='claim_count',
    feature_names=feature_names,
    link_fn='poisson',
    glm_preds_col='glm_predictions',
    glm_df=glm_df,
    mapping_dict=mapping_dict,
    shap_df=mp.shap_df,  # Reuse computed SHAP values
    verbose=False
)

# Customize colors
mp_custom.config['colors']['shap'] = '#e41a1c'       # Red
mp_custom.config['colors']['glm'] = '#4daf4a'        # Green
mp_custom.config['colors']['weight'] = '#984ea3'     # Purple
mp_custom.config['colors']['actuals'] = '#ff7f00'    # Orange

# Customize labels
mp_custom.config['labels']['shap'] = 'GBM Effect'
mp_custom.config['labels']['glm'] = 'GLM Factor'

# Customize line widths
mp_custom.config['line_width']['shap'] = 3
mp_custom.config['line_width']['glm'] = 3

print("Custom configuration applied!")

In [ ]:
# Plot with custom styling
mp_custom.univariate_plot(
    var_name='age',
    shap=True,
    glm=True,
    actuals=True,
    weight=True,
    plot_name='Age - Custom Styled Plot'
)

## 8. Plot Engines

MintyPython supports two plot engines: Bokeh (interactive) and Matplotlib (static).

### 8.1 Bokeh Engine (Interactive)

In [ ]:
# Bokeh is the default engine - produces interactive HTML plots
mp.univariate_plot(
    var_name='years_licensed',
    shap=True,
    glm=True,
    weight=True,
    engine='Bokeh',
    plot_name='Years Licensed - Bokeh (Interactive)'
)

### 8.2 Matplotlib Engine (Static)

In [ ]:
# Matplotlib produces static plots - useful for reports and PDFs
mp.univariate_plot(
    var_name='years_licensed',
    shap=True,
    glm=True,
    weight=True,
    engine='Matplotlib',
    plot_name='Years Licensed - Matplotlib (Static)'
)

## Summary

This demo covered:

1. **Data Generation**: Creating synthetic insurance data with realistic feature relationships
2. **Model Training**: XGBoost with Poisson objective for claim frequency modeling
3. **MintyPython Initialization**: Setting up the library with models, data, and GLM comparisons
4. **Univariate Plots**: Various ways to visualize single-feature effects
5. **Bivariate Plots**: Visualizing feature interactions
6. **Model Comparison**: Comparing multiple GBM models side by side
7. **Customization**: Changing colors, labels, and styling
8. **Plot Engines**: Using Bokeh for interactive plots and Matplotlib for static output

For more details, see the MintyPython documentation and the CLAUDE.md file in the repository root.